# Assignment 3 — Preprocessing & XGBoost Modeling

EDA 노트북(`01_train_set_eda.ipynb`) 결과를 바탕으로 전처리 파이프라인과 **XGBoost** 모델을 셀 단위로 구성한다.

분석 목표:
1. 시각 → 4분할 시간대 버킷 변환, 결측 플래그, 파생 변수 생성
2. 누수(Leakage) 방지를 위한 fold 내 인코더 fit (Frequency / OOF Target Encoding)
3. Stratified 5-Fold CV 로 **Log-loss** 측정 (과제 채점 기준)
4. 하이퍼파라미터 튜닝 + (선택) 준지도 학습으로 미라벨 데이터 활용
5. test set 예측 확률 산출 → 제출 CSV 작성

> **규칙**
> - 하드코딩 금지: 컬럼 그룹·시간대 버킷 경계·임계값은 모두 상단 설정 변수로 관리
> - CV 필수 (과제 명시)
> - 외부 데이터 사용 금지 (train.csv 정보만으로 인코딩/대치)

## 1. 환경 설정 및 데이터 로드

In [ ]:
# >>> [분석 노트]
# 라이브러리 로드 및 표시 옵션·시드 고정.
# - pandas/numpy: 데이터 처리
# - sklearn: KFold, metrics, preprocessing
# - xgboost: 메인 모델
# - RANDOM_STATE 로 fold 분할/모델 시드 통일 (재현성)
# <<<
import os
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import log_loss
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("xgboost:", xgb.__version__)

In [ ]:
# >>> [분석 노트]
# 데이터 경로/파일명/컬럼 그룹/임계값을 모두 설정 객체로 집중 (하드코딩 금지).
# - CONFIG: 단일 진입점, 다른 셀에서는 CONFIG.* 만 참조
# - 시간대 버킷 경계·결측 플래그 대상 컬럼·CV fold 수 등 모두 변수화
# - 환경변수로 데이터 위치/파일명 오버라이드 가능
# <<<
def resolve_data_dir() -> Path:
    env = os.environ.get("ASSIGNMENT3_DATA_DIR")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd()
    for cand in [here / "data", here.parent / "data", here]:
        if cand.exists():
            return cand.resolve()
    return here.resolve()


def resolve_csv(data_dir: Path, exact: str, keyword: str) -> Path:
    direct = data_dir / exact
    if direct.exists():
        return direct.resolve()
    matches = sorted(p for p in data_dir.glob("*.csv") if keyword.lower() in p.name.lower())
    if not matches:
        raise FileNotFoundError(
            f"Could not find {exact!r} or any csv containing {keyword!r} under {data_dir}"
        )
    return matches[0].resolve()


@dataclass
class Config:
    # --- 경로 ---
    data_dir: Path = field(default_factory=resolve_data_dir)
    train_filename: str = os.environ.get("ASSIGNMENT3_TRAIN", "train_set.csv")
    test_filename: str = os.environ.get("ASSIGNMENT3_TEST", "test_set.csv")
    submission_dir: Path = field(default_factory=lambda: Path.cwd() / "submissions")
    submission_filename: str = os.environ.get(
        "ASSIGNMENT3_SUBMISSION", "ML_HW3_test.csv"
    )

    # --- 컬럼 ---
    id_col: str = "ID"
    target_col: str = "Delay"
    positive_label: str = "Delayed"
    negative_label: str = "Not_Delayed"

    # 단일값/중복 제거 컬럼
    drop_cols: tuple = ("Cancelled", "Diverted", "Airline")

    # 원본 시각 컬럼 (파싱 후 drop)
    time_cols: tuple = ("Estimated_Departure_Time", "Estimated_Arrival_Time")

    # 결측 플래그 대상 (EDA에서 결측 약 10.9%로 확인된 컬럼)
    missing_flag_cols: tuple = (
        "Estimated_Departure_Time",
        "Estimated_Arrival_Time",
        "Origin_State",
        "Destination_State",
        "Carrier_Code(IATA)",
        "Carrier_ID(DOT)",
        "Tail_Number",
    )

    # 인코딩 대상 (Frequency + OOF Target)
    freq_target_cols: tuple = (
        "Tail_Number",
        "Origin_Airport",
        "Destination_Airport",
        "Route",
        "Carrier_Route",
    )
    # 인코딩 대상 (OOF Target only)
    target_only_cols: tuple = ("Origin_State", "Destination_State")
    # 인코딩 대상 (Ordinal)
    ordinal_cols: tuple = ("Carrier_Code(IATA)", "Carrier_ID(DOT)")

    # 시간대 버킷 경계 (시 단위, 양 끝 포함)
    time_buckets: tuple = (
        ("새벽", 0, 5),
        ("오전", 6, 11),
        ("오후", 12, 17),
        ("저녁", 18, 23),
    )
    bucket_missing_label: str = "__missing__"

    # --- 학습 ---
    n_splits: int = 5
    target_smoothing: float = 20.0   # OOF target encoder smoothing
    pseudo_threshold: float = 0.9    # 준지도 confidence 임계값 (양/음 모두 ≥ 이 값)


CONFIG = Config()

TRAIN_PATH = resolve_csv(CONFIG.data_dir, CONFIG.train_filename, "train")
TEST_PATH = resolve_csv(CONFIG.data_dir, CONFIG.test_filename, "test")

print("DATA_DIR :", CONFIG.data_dir)
print("TRAIN    :", TRAIN_PATH.name)
print("TEST     :", TEST_PATH.name)
print("SUBMIT   :", CONFIG.submission_dir / CONFIG.submission_filename)

In [ ]:
# >>> [분석 노트]
# train/test 로드 후 shape·결측 비율 확인 (sanity).
# - test set 의 ID 는 제출 파일 정렬용으로 보관
# <<<
raw_train = pd.read_csv(TRAIN_PATH)
raw_test = pd.read_csv(TEST_PATH)
test_ids = raw_test[CONFIG.id_col].copy()

print(f"train shape: {raw_train.shape}")
print(f"test  shape: {raw_test.shape}")
print(f"label 결측 비율: {raw_train[CONFIG.target_col].isna().mean()*100:.2f} %")

## 2. 전처리 — 컬럼 정리

- `Cancelled`, `Diverted`: train 전체 0 단일값 → drop
- `Airline`: `Carrier_ID(DOT)`와 1:1 매핑 → drop (Carrier_ID 유지)
- `ID`: 학습 비사용 (test 제출용으로는 별도 보관)

In [ ]:
# >>> [분석 노트]
# CONFIG.drop_cols 에 정의된 컬럼만 일괄 제거 (하드코딩 금지).
# - 존재하지 않는 컬럼은 무시 (errors='ignore')
# - ID 는 학습 피처에서는 제외하되 test 예측 정렬용으로 별도 보관됨
# <<<
def drop_columns(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    return df.drop(columns=[c for c in cols if c in df.columns], errors="ignore")


train_df = drop_columns(raw_train, CONFIG.drop_cols).copy()
test_df = drop_columns(raw_test, CONFIG.drop_cols).copy()

# ID 는 학습 단계에서 제외 (test 의 ID 는 위에서 test_ids 로 보관됨)
train_df = train_df.drop(columns=[CONFIG.id_col], errors="ignore")
test_df = test_df.drop(columns=[CONFIG.id_col], errors="ignore")

print("train columns:", list(train_df.columns))
print("test  columns:", list(test_df.columns))

## 3. 전처리 — 시각 → 시간대 버킷 + Duration

- 원본 `HHMM.0` float 에서 **시 단위만** 추출, 분은 폐기
- 4분할 버킷(`새벽 / 오전 / 오후 / 저녁`) — 경계는 `CONFIG.time_buckets`
- `Estimated_Duration_min`: `(Arr_Hour - Dep_Hour) mod 24 * 60`
- 이상치(`hour ≥ 24`)는 `__missing__`

In [ ]:
# >>> [분석 노트]
# HHMM.0 → 시 단위 정수 (0~23). 비정상값/결측은 NaN.
# - 분 단위 정밀도는 노이즈로 판단해 버림
# - 분 부분이 60 이상이거나 시가 24 이상이면 결측 처리
# <<<
def extract_hour(value: float) -> float:
    if pd.isna(value):
        return np.nan
    v = int(value)
    h, m = divmod(v, 100)
    if not (0 <= h <= 23 and 0 <= m <= 59):
        return np.nan
    return float(h)


def bucketize_hour(hour: float, buckets) -> str:
    if pd.isna(hour):
        return CONFIG.bucket_missing_label
    h = int(hour)
    for name, lo, hi in buckets:
        if lo <= h <= hi:
            return name
    return CONFIG.bucket_missing_label


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dep_col, arr_col = CONFIG.time_cols
    dep_hour = out[dep_col].apply(extract_hour)
    arr_hour = out[arr_col].apply(extract_hour)

    out["Dep_Hour"] = dep_hour
    out["Arr_Hour"] = arr_hour
    out["Dep_TimeBucket"] = dep_hour.apply(lambda h: bucketize_hour(h, CONFIG.time_buckets))
    out["Arr_TimeBucket"] = arr_hour.apply(lambda h: bucketize_hour(h, CONFIG.time_buckets))

    duration_h = (arr_hour - dep_hour) % 24
    out["Estimated_Duration_min"] = duration_h * 60

    # 원본 시각 컬럼 drop (정보 중복)
    out = out.drop(columns=list(CONFIG.time_cols), errors="ignore")
    return out


train_df = add_time_features(train_df)
test_df = add_time_features(test_df)

print("버킷 분포 (train):")
print(train_df["Dep_TimeBucket"].value_counts(dropna=False))
print("\nEstimated_Duration_min describe:")
print(train_df["Estimated_Duration_min"].describe())

## 4. 전처리 — 결측 플래그

EDA에서 결측 자체의 타깃 신호는 Δ<0.006 으로 미약했으나, 트리 모델에 잔존 신호 전달용으로 추가.

In [ ]:
# >>> [분석 노트]
# CONFIG.missing_flag_cols 에 대해 <col>_is_missing (0/1) 플래그 생성.
# - 원본 컬럼은 유지 (인코딩 단계에서 그대로 사용)
# - 컬럼이 데이터프레임에 존재하는 경우에만 처리 (시각 컬럼은 이미 파생됐을 수 있음)
# <<<
def add_missing_flags(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        # 원본 시각 컬럼은 add_time_features 에서 drop 됐을 수 있으므로
        # 파생된 시 단위 컬럼(Dep_Hour/Arr_Hour) 로 대체
        source = c
        if c == "Estimated_Departure_Time":
            source = "Dep_Hour"
        elif c == "Estimated_Arrival_Time":
            source = "Arr_Hour"
        if source in out.columns:
            out[f"{c}_is_missing"] = out[source].isna().astype(int)
    return out


train_df = add_missing_flags(train_df, CONFIG.missing_flag_cols)
test_df = add_missing_flags(test_df, CONFIG.missing_flag_cols)

missing_flag_cols = [c for c in train_df.columns if c.endswith("_is_missing")]
print("생성된 결측 플래그:", missing_flag_cols)
print("\nflag 평균 (train):")
print(train_df[missing_flag_cols].mean().round(4))

## 5. 전처리 — 파생 변수

- `Is_Summer`: Month ∈ {6,7,8} — EDA에서 지연율 0.20~0.22
- `Is_FallLow`: Month ∈ {9,10} — EDA에서 지연율 0.14~0.15
- `Route`: `Origin_Airport + "_" + Destination_Airport`
- `Carrier_Route`: `Carrier_ID(DOT) + Route`

In [ ]:
# >>> [분석 노트]
# 도메인/EDA 기반 파생 변수 생성.
# - 계절 플래그는 Month set 으로 정의 (CONFIG 에 두지 않은 이유: 도메인 정의가 명백 + EDA 직접 산출)
# - Route/Carrier_Route 는 문자열 결합. NaN 은 '__missing__' 으로 보존
# <<<
SUMMER_MONTHS = {6, 7, 8}
FALL_LOW_MONTHS = {9, 10}


def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Is_Summer"] = out["Month"].isin(SUMMER_MONTHS).astype(int)
    out["Is_FallLow"] = out["Month"].isin(FALL_LOW_MONTHS).astype(int)

    o = out["Origin_Airport"].fillna(CONFIG.bucket_missing_label).astype(str)
    d = out["Destination_Airport"].fillna(CONFIG.bucket_missing_label).astype(str)
    out["Route"] = o + "_" + d

    c = out["Carrier_ID(DOT)"].fillna(-1).astype(int).astype(str)
    out["Carrier_Route"] = c + "_" + out["Route"]
    return out


train_df = add_derived(train_df)
test_df = add_derived(test_df)

print("파생 컬럼 카디널리티 (train):")
for c in ["Is_Summer", "Is_FallLow", "Route", "Carrier_Route"]:
    print(f"  {c}: {train_df[c].nunique()}")

## 6. 전처리 — 타깃 처리 및 라벨/언라벨 분리

In [ ]:
# >>> [분석 노트]
# 타깃 매핑: Not_Delayed → 0, Delayed → 1. NaN 은 미라벨로 분리.
# - labeled_df: 모델 학습/CV 대상 (~210K)
# - unlabeled_df: 준지도 학습 후보 (~590K)
# - test_df_X: test set 피처 (예측 대상)
# <<<
LABEL_MAP = {CONFIG.negative_label: 0, CONFIG.positive_label: 1}

y_full = train_df[CONFIG.target_col].map(LABEL_MAP)
X_full = train_df.drop(columns=[CONFIG.target_col])

labeled_mask = y_full.notna()
labeled_df = X_full.loc[labeled_mask].reset_index(drop=True)
y_labeled = y_full.loc[labeled_mask].astype(int).reset_index(drop=True)
unlabeled_df = X_full.loc[~labeled_mask].reset_index(drop=True)
test_df_X = test_df.copy()

print(f"labeled  : {labeled_df.shape}")
print(f"unlabeled: {unlabeled_df.shape}")
print(f"test     : {test_df_X.shape}")
print(f"\nbase rate (Delayed): {y_labeled.mean():.4f}")
print("\n클래스 분포:")
print(y_labeled.value_counts(normalize=True).round(4))

## 7. 전처리 — 인코더 정의 (fold 내 fit 대상)

- `FrequencyEncoder`: 학습 fold 의 카테고리 등장 빈도
- `OOFTargetEncoder`: 학습 fold 의 카테고리별 타깃 평균 (smoothing 적용, 누수 방지)
- `OrdinalEncoder` (lightweight): 학습 fold 의 고유값 → 정수

In [ ]:
# >>> [분석 노트]
# 누수(Leakage) 방지를 위해 fold 내부에서 fit 하는 경량 인코더 3종.
# - FrequencyEncoder: 카테고리 → 빈도 (train fold 기준). unseen 은 0
# - OOFTargetEncoder: 카테고리 → 타깃 평균. smoothing 으로 희소 카테고리 안정화
# - OrdinalEncoder: 카테고리 → 정수 (train fold 기준). unseen 은 -1
# <<<
class FrequencyEncoder:
    def __init__(self):
        self.freq_: dict = {}

    def fit(self, s: pd.Series):
        self.freq_ = s.fillna(CONFIG.bucket_missing_label).astype(str).value_counts().to_dict()
        return self

    def transform(self, s: pd.Series) -> pd.Series:
        return s.fillna(CONFIG.bucket_missing_label).astype(str).map(self.freq_).fillna(0).astype(float)


class OOFTargetEncoder:
    def __init__(self, smoothing: float = 20.0):
        self.smoothing = smoothing
        self.mean_: dict = {}
        self.global_mean_: float = 0.0

    def fit(self, s: pd.Series, y: pd.Series):
        s = s.fillna(CONFIG.bucket_missing_label).astype(str)
        self.global_mean_ = float(y.mean())
        agg = pd.DataFrame({"x": s, "y": y.values}).groupby("x")["y"].agg(["mean", "count"])
        smooth = (agg["mean"] * agg["count"] + self.global_mean_ * self.smoothing) / (agg["count"] + self.smoothing)
        self.mean_ = smooth.to_dict()
        return self

    def transform(self, s: pd.Series) -> pd.Series:
        return s.fillna(CONFIG.bucket_missing_label).astype(str).map(self.mean_).fillna(self.global_mean_).astype(float)


class OrdinalEncoder:
    def __init__(self):
        self.mapping_: dict = {}

    def fit(self, s: pd.Series):
        uniq = s.fillna(CONFIG.bucket_missing_label).astype(str).unique()
        self.mapping_ = {v: i for i, v in enumerate(sorted(uniq))}
        return self

    def transform(self, s: pd.Series) -> pd.Series:
        return s.fillna(CONFIG.bucket_missing_label).astype(str).map(self.mapping_).fillna(-1).astype(int)


# 시간대 버킷도 ordinal 로 처리 (XGBoost 는 범주형 직접 지원이 약하므로 수치화 필요)
BUCKET_COLS = ("Dep_TimeBucket", "Arr_TimeBucket")


def encode_train_valid(
    X_tr: pd.DataFrame, y_tr: pd.Series,
    X_va: pd.DataFrame,
) -> tuple:
    """학습 fold 에서 인코더 fit, valid 에 transform 적용. test 변환을 위한 인코더도 반환."""
    X_tr_enc = X_tr.copy()
    X_va_enc = X_va.copy()
    encoders = {"freq": {}, "target": {}, "ordinal": {}}

    # Frequency + OOF Target
    for col in CONFIG.freq_target_cols:
        fe = FrequencyEncoder().fit(X_tr[col])
        te = OOFTargetEncoder(CONFIG.target_smoothing).fit(X_tr[col], y_tr)
        X_tr_enc[f"{col}__freq"] = fe.transform(X_tr[col])
        X_va_enc[f"{col}__freq"] = fe.transform(X_va[col])
        X_tr_enc[f"{col}__te"] = te.transform(X_tr[col])
        X_va_enc[f"{col}__te"] = te.transform(X_va[col])
        encoders["freq"][col] = fe
        encoders["target"][col] = te
        X_tr_enc = X_tr_enc.drop(columns=[col])
        X_va_enc = X_va_enc.drop(columns=[col])

    # OOF Target only
    for col in CONFIG.target_only_cols:
        te = OOFTargetEncoder(CONFIG.target_smoothing).fit(X_tr[col], y_tr)
        X_tr_enc[f"{col}__te"] = te.transform(X_tr[col])
        X_va_enc[f"{col}__te"] = te.transform(X_va[col])
        encoders["target"][col] = te
        X_tr_enc = X_tr_enc.drop(columns=[col])
        X_va_enc = X_va_enc.drop(columns=[col])

    # Ordinal (저카디널리티 또는 ID 형태)
    for col in tuple(CONFIG.ordinal_cols) + BUCKET_COLS:
        oe = OrdinalEncoder().fit(X_tr[col])
        X_tr_enc[col] = oe.transform(X_tr[col])
        X_va_enc[col] = oe.transform(X_va[col])
        encoders["ordinal"][col] = oe

    return X_tr_enc, X_va_enc, encoders


def apply_encoders(X: pd.DataFrame, encoders: dict) -> pd.DataFrame:
    out = X.copy()
    for col, fe in encoders["freq"].items():
        out[f"{col}__freq"] = fe.transform(X[col])
    for col, te in encoders["target"].items():
        out[f"{col}__te"] = te.transform(X[col])
    for col, oe in encoders["ordinal"].items():
        out[col] = oe.transform(X[col])
    # freq_target_cols / target_only_cols 의 원본은 drop (이미 인코딩됨)
    drop_cols = list(CONFIG.freq_target_cols) + list(CONFIG.target_only_cols)
    out = out.drop(columns=[c for c in drop_cols if c in out.columns])
    return out


print("인코더 정의 완료")

## 8. 전처리 — 단일 split sanity 검증

CV 들어가기 전, 한 번의 stratified split 으로 인코딩 결과(컬럼·dtype·결측)를 점검.

In [ ]:
# >>> [분석 노트]
# 단일 80/20 stratified split 으로 인코딩 파이프라인 정상 동작 검증.
# - 컬럼 일치 / dtype object 잔존 / 결측 비율 점검
# - XGBoost 는 NaN 처리 가능하지만 인코딩 단계 결과는 NaN 없어야 함 (Distance/Duration 제외)
# <<<
X_tr_s, X_va_s, y_tr_s, y_va_s = train_test_split(
    labeled_df, y_labeled,
    test_size=0.2, stratify=y_labeled, random_state=RANDOM_STATE,
)

X_tr_enc_s, X_va_enc_s, encoders_s = encode_train_valid(X_tr_s, y_tr_s, X_va_s)

print("X_tr_enc shape:", X_tr_enc_s.shape)
print("X_va_enc shape:", X_va_enc_s.shape)
print("\n컬럼 일치:", list(X_tr_enc_s.columns) == list(X_va_enc_s.columns))
print("\ndtype 분포:")
print(X_tr_enc_s.dtypes.value_counts())
print("\n결측 비율 상위 10:")
print((X_tr_enc_s.isna().mean() * 100).round(2).sort_values(ascending=False).head(10))

## 9. 모델 1 — XGBoost 베이스라인 (단일 split)

기본 파라미터로 빠르게 baseline Log-loss 확인. 이후 CV / 튜닝의 비교 기준.

In [ ]:
# >>> [분석 노트]
# XGBoost 베이스라인: objective=binary:logistic, eval_metric=logloss.
# - tree_method='hist' 로 대규모 데이터 빠르게 학습
# - early_stopping_rounds 로 validation logloss 기준 조기 종료
# - scale_pos_weight 는 아직 적용하지 않음 (Log-loss 채점에선 확률 보정이 중요 → 강제 보정 지양)
# <<<
BASELINE_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "learning_rate": 0.1,
    "max_depth": 6,
    "min_child_weight": 1.0,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}
BASELINE_NUM_BOOST_ROUND = 2000
EARLY_STOPPING_ROUNDS = 50

dtrain_s = xgb.DMatrix(X_tr_enc_s, label=y_tr_s)
dvalid_s = xgb.DMatrix(X_va_enc_s, label=y_va_s)

booster_s = xgb.train(
    params=BASELINE_PARAMS,
    dtrain=dtrain_s,
    num_boost_round=BASELINE_NUM_BOOST_ROUND,
    evals=[(dtrain_s, "train"), (dvalid_s, "valid")],
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    verbose_eval=100,
)

valid_pred_s = booster_s.predict(dvalid_s, iteration_range=(0, booster_s.best_iteration + 1))
baseline_ll = log_loss(y_va_s, valid_pred_s)
print(f"\n[Baseline] Best iter = {booster_s.best_iteration}, Valid Log-loss = {baseline_ll:.5f}")

## 10. 모델 2 — Stratified 5-Fold CV (과제 필수)

fold 내부에서 인코더 fit → leakage 방지. OOF Log-loss 산출.

In [ ]:
# >>> [분석 노트]
# Stratified K-Fold CV. 각 fold 내부에서:
# 1) 인코더 fit (train fold 만 사용)
# 2) XGBoost 학습 (valid fold 로 early stopping)
# 3) OOF 예측 누적
# 마지막에 OOF 전체 Log-loss 산출.
# <<<
def run_cv(params: dict, X: pd.DataFrame, y: pd.Series,
           n_splits: int = CONFIG.n_splits,
           num_boost_round: int = BASELINE_NUM_BOOST_ROUND,
           early_stopping_rounds: int = EARLY_STOPPING_ROUNDS,
           verbose: bool = False) -> dict:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(X), dtype=float)
    fold_scores = []
    best_iters = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr = X.iloc[tr_idx].reset_index(drop=True)
        X_va = X.iloc[va_idx].reset_index(drop=True)
        y_tr = y.iloc[tr_idx].reset_index(drop=True)
        y_va = y.iloc[va_idx].reset_index(drop=True)

        X_tr_enc, X_va_enc, _ = encode_train_valid(X_tr, y_tr, X_va)

        dtr = xgb.DMatrix(X_tr_enc, label=y_tr)
        dva = xgb.DMatrix(X_va_enc, label=y_va)

        booster = xgb.train(
            params=params,
            dtrain=dtr,
            num_boost_round=num_boost_round,
            evals=[(dva, "valid")],
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=200 if verbose else False,
        )
        pred = booster.predict(dva, iteration_range=(0, booster.best_iteration + 1))
        oof[va_idx] = pred
        fold_ll = log_loss(y_va, pred)
        fold_scores.append(fold_ll)
        best_iters.append(booster.best_iteration)
        print(f"  Fold {fold}: best_iter={booster.best_iteration:4d}  log-loss={fold_ll:.5f}")

    overall_ll = log_loss(y, oof)
    print(f"  ---")
    print(f"  Fold mean log-loss : {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")
    print(f"  OOF overall log-loss: {overall_ll:.5f}")
    print(f"  Mean best_iter      : {int(np.mean(best_iters))}")
    return {
        "oof": oof,
        "fold_scores": fold_scores,
        "overall": overall_ll,
        "mean_best_iter": int(np.mean(best_iters)),
    }


print("[CV] Baseline params")
cv_baseline = run_cv(BASELINE_PARAMS, labeled_df, y_labeled)

## 11. 모델 3 — 하이퍼파라미터 튜닝

작은 grid 위에서 CV Log-loss 기준 최적 조합 탐색. (시간 절약을 위해 핵심 파라미터만)

In [ ]:
# >>> [분석 노트]
# 소규모 grid search (CV 기준).
# - 변경 파라미터: learning_rate, max_depth, min_child_weight, subsample/colsample, reg_lambda
# - 전체 grid 폭은 의도적으로 작게 (학습시간/현실성)
# - 각 후보를 run_cv 로 평가하고 best 조합 저장
# <<<
PARAM_GRID = [
    {"learning_rate": 0.05, "max_depth": 6, "min_child_weight": 1.0,
     "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1.0},
    {"learning_rate": 0.05, "max_depth": 8, "min_child_weight": 5.0,
     "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0},
    {"learning_rate": 0.05, "max_depth": 7, "min_child_weight": 3.0,
     "subsample": 0.85, "colsample_bytree": 0.85, "reg_lambda": 2.0},
    {"learning_rate": 0.03, "max_depth": 8, "min_child_weight": 5.0,
     "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 2.0},
]

results = []
for i, override in enumerate(PARAM_GRID, 1):
    params = {**BASELINE_PARAMS, **override}
    print(f"\n[CV {i}/{len(PARAM_GRID)}] override = {override}")
    res = run_cv(params, labeled_df, y_labeled)
    results.append({"override": override, "params": params, **res})

results = sorted(results, key=lambda r: r["overall"])
best = results[0]

print("\n=== Tuning summary ===")
for r in results:
    print(f"  ll={r['overall']:.5f}  best_iter={r['mean_best_iter']}  override={r['override']}")
print(f"\nBest CV log-loss: {best['overall']:.5f}")

## 12. 모델 4 — 준지도 학습 (Pseudo-Labeling)

1. best 모델의 unlabeled 예측 산출
2. confidence ≥ `CONFIG.pseudo_threshold` (또는 ≤ 1 - threshold) 인 행만 pseudo-label 부여
3. labeled + pseudo-labeled 합쳐 CV 재평가
4. Log-loss 개선되면 채택, 아니면 베이스 유지

In [ ]:
# >>> [분석 노트]
# 준지도 학습:
# 1) best 파라미터로 labeled 전체에 fit → unlabeled 예측
# 2) 양극단(>=0.9 또는 <=0.1) 만 pseudo-label 부여
# 3) labeled + pseudo 로 CV 재실행해 Log-loss 비교
# 4) 개선되지 않으면 best 파라미터 그대로 사용
# 주의: pseudo set 자체는 CV 안에서 valid fold 로 평가하지 않음 (라벨이 모델 예측이므로 평가 신뢰성 없음)
#        → labeled 만 stratify 대상으로 두고, pseudo 는 train fold 에만 합쳐 학습
# <<<
def make_pseudo_labels(params: dict, X_lab, y_lab, X_unl,
                       threshold: float, num_boost_round: int) -> tuple:
    X_tr_enc, X_unl_enc, encoders = encode_train_valid(X_lab, y_lab, X_unl)
    dtr = xgb.DMatrix(X_tr_enc, label=y_lab)
    booster = xgb.train(params=params, dtrain=dtr, num_boost_round=num_boost_round)
    dunl = xgb.DMatrix(X_unl_enc)
    probs = booster.predict(dunl)

    high = probs >= threshold
    low = probs <= (1 - threshold)
    keep = high | low
    pseudo_y = pd.Series(np.where(probs >= 0.5, 1, 0), index=X_unl.index)
    pseudo_X = X_unl.loc[keep].reset_index(drop=True)
    pseudo_y = pseudo_y.loc[keep].reset_index(drop=True)
    print(f"  pseudo kept: {keep.sum():,} / {len(probs):,} ({keep.mean()*100:.2f}%)")
    print(f"  pseudo class ratio: {pseudo_y.mean():.4f}")
    return pseudo_X, pseudo_y


def run_cv_with_pseudo(params, X_lab, y_lab, pseudo_X, pseudo_y,
                       n_splits=CONFIG.n_splits,
                       num_boost_round=BASELINE_NUM_BOOST_ROUND,
                       early_stopping_rounds=EARLY_STOPPING_ROUNDS):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(X_lab), dtype=float)
    fold_scores = []
    best_iters = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_lab, y_lab), 1):
        X_tr = X_lab.iloc[tr_idx].reset_index(drop=True)
        y_tr = y_lab.iloc[tr_idx].reset_index(drop=True)
        X_va = X_lab.iloc[va_idx].reset_index(drop=True)
        y_va = y_lab.iloc[va_idx].reset_index(drop=True)

        # train fold + pseudo 합치기 (valid fold 는 그대로 유지)
        X_tr_plus = pd.concat([X_tr, pseudo_X], ignore_index=True)
        y_tr_plus = pd.concat([y_tr, pseudo_y], ignore_index=True)

        X_tr_enc, X_va_enc, _ = encode_train_valid(X_tr_plus, y_tr_plus, X_va)
        dtr = xgb.DMatrix(X_tr_enc, label=y_tr_plus)
        dva = xgb.DMatrix(X_va_enc, label=y_va)
        booster = xgb.train(
            params=params, dtrain=dtr, num_boost_round=num_boost_round,
            evals=[(dva, "valid")],
            early_stopping_rounds=early_stopping_rounds, verbose_eval=False,
        )
        pred = booster.predict(dva, iteration_range=(0, booster.best_iteration + 1))
        oof[va_idx] = pred
        fold_ll = log_loss(y_va, pred)
        fold_scores.append(fold_ll)
        best_iters.append(booster.best_iteration)
        print(f"  Fold {fold}: best_iter={booster.best_iteration:4d}  log-loss={fold_ll:.5f}")

    overall_ll = log_loss(y_lab, oof)
    print(f"  OOF overall log-loss: {overall_ll:.5f}")
    return {"oof": oof, "fold_scores": fold_scores, "overall": overall_ll,
            "mean_best_iter": int(np.mean(best_iters))}


print("\n[Pseudo] generate pseudo-labels with best params")
pseudo_X, pseudo_y = make_pseudo_labels(
    best["params"], labeled_df, y_labeled, unlabeled_df,
    threshold=CONFIG.pseudo_threshold,
    num_boost_round=best["mean_best_iter"],
)

print("\n[CV with pseudo]")
cv_pseudo = run_cv_with_pseudo(
    best["params"], labeled_df, y_labeled, pseudo_X, pseudo_y,
)

use_pseudo = cv_pseudo["overall"] < best["overall"]
print(f"\nbest (no pseudo): {best['overall']:.5f}")
print(f"with pseudo     : {cv_pseudo['overall']:.5f}")
print(f"→ use_pseudo = {use_pseudo}")

## 13. 모델 5 — 최종 학습 및 test 예측

- best 파라미터 + (use_pseudo 가 True면) pseudo set 포함
- 학습 라운드 수는 CV에서 측정된 평균 best_iter 사용
- test 예측 → 확률 산출

In [ ]:
# >>> [분석 노트]
# 최종 학습:
# - 전체 labeled (+ 선택적으로 pseudo) 로 fit
# - num_boost_round 는 CV 평균 best_iter (early stopping 없이 고정 round)
# - test 에 대해 확률 예측 → Not_Delayed / Delayed 두 컬럼 산출
# <<<
if use_pseudo:
    X_final = pd.concat([labeled_df, pseudo_X], ignore_index=True)
    y_final = pd.concat([y_labeled, pseudo_y], ignore_index=True)
    num_rounds_final = cv_pseudo["mean_best_iter"]
    print("최종 학습: labeled + pseudo")
else:
    X_final = labeled_df.copy()
    y_final = y_labeled.copy()
    num_rounds_final = best["mean_best_iter"]
    print("최종 학습: labeled only")

print(f"X_final shape: {X_final.shape}, num_rounds_final: {num_rounds_final}")

# 인코더는 X_final 전체로 fit (test 도 동일 인코더 사용)
X_final_enc, X_test_enc, final_encoders = encode_train_valid(X_final, y_final, test_df_X)

dfinal = xgb.DMatrix(X_final_enc, label=y_final)
dtest = xgb.DMatrix(X_test_enc)

final_booster = xgb.train(
    params=best["params"],
    dtrain=dfinal,
    num_boost_round=max(num_rounds_final, 1),
    evals=[(dfinal, "train")],
    verbose_eval=200,
)

test_proba_delayed = final_booster.predict(dtest)
print(f"\ntest 예측 통계: mean={test_proba_delayed.mean():.4f}, min={test_proba_delayed.min():.4f}, max={test_proba_delayed.max():.4f}")

## 14. 제출 파일 생성

- 헤더: `ID,Not_Delayed,Delayed`
- 각 행 두 확률의 합 = 1
- 파일명: `CONFIG.submission_filename` (기본 `ML_HW3_test.csv`, 환경변수로 학번/이름 치환 가능)

In [ ]:
# >>> [분석 노트]
# 제출 CSV 작성.
# - test_ids 순서 유지 (test_df_X 와 동일 순서로 예측됐는지 sanity 체크)
# - 확률 합 = 1 검증
# - 제출 디렉터리 없으면 생성
# <<<
assert len(test_ids) == len(test_proba_delayed), "ID 와 예측 수 불일치"

submission = pd.DataFrame({
    CONFIG.id_col: test_ids.values,
    CONFIG.negative_label: 1.0 - test_proba_delayed,
    CONFIG.positive_label: test_proba_delayed,
})

row_sums = submission[[CONFIG.negative_label, CONFIG.positive_label]].sum(axis=1)
assert np.allclose(row_sums, 1.0), "행별 확률 합이 1이 아님"

CONFIG.submission_dir.mkdir(parents=True, exist_ok=True)
submission_path = CONFIG.submission_dir / CONFIG.submission_filename
submission.to_csv(submission_path, index=False)

print(f"제출 파일 작성 완료: {submission_path}")
print(f"shape: {submission.shape}")
print(submission.head())